# Agentes com LangChain, Ollama e Qwen

Nesta aula vamos construir um agente que recebe uma pergunta, interpreta a intenção, escolhe uma ferramenta e executa a ação.

```text
Usuário → Agente/Qwen → Tool → Resultado → Qwen → Resposta
```

Usaremos Ollama + Qwen 3 4B, sem API paga da OpenAI.


## 1. O que é um agente?

Um LLM sozinho normalmente faz:

```text
Pergunta → LLM → Resposta
```

Um agente adiciona a capacidade de decidir quais ações executar:

```text
Pergunta
   ↓
Agente
   ↓
Escolhe uma Tool
   ↓
Executa
   ↓
Resultado
   ↓
Resposta
```

Uma forma simples de pensar:

```text
Agente = LLM + ferramentas + processo de decisão
```


## 2. Tool x Agente

Uma **Tool** é uma função que o agente pode executar.

O **Agente** decide quando e como utilizar as Tools.

Exemplo:

```text
Pergunta
   ↓
Agente
   ├── consultar_horario()
   ├── criar_tarefa()
   └── listar_tarefas()
```


## 3. Dependência

Para este exemplo vamos utilizar LangGraph.

Adicione ao `requirements.txt`:

```text
langgraph
```

Depois:

```powershell
pip install -r requirements.txt
```


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent


## 4. Configurando o Qwen

O modelo será responsável por interpretar a pergunta e decidir qual ferramenta utilizar.


In [ ]:
llm = ChatOllama(
    model="qwen3:4b",
    base_url="http://localhost:11434"
)


## 5. Criando uma Tool

O decorator `@tool` transforma uma função Python em uma ferramenta disponível para o agente.

A descrição da função também é importante: o agente utiliza essa informação para entender quando a ferramenta deve ser usada.


In [ ]:
@tool
def consultar_horario() -> str:
    """Consulta o horário atual do sistema."""
    from datetime import datetime

    horario = datetime.now().strftime("%H:%M:%S")
    return f"O horário atual é {horario}."


## 6. Criando Tools para tarefas

Agora criaremos ferramentas que simulam operações de um sistema de tarefas.

Em um projeto real, essas funções poderiam chamar o Task Service.


In [ ]:
@tool
def criar_tarefa(titulo: str) -> str:
    """Cria uma tarefa com o título informado pelo usuário."""
    return f'Tarefa "{titulo}" criada com sucesso.'


@tool
def listar_tarefas() -> str:
    """Lista as tarefas cadastradas no sistema."""
    tarefas = [
        "Estudar LangChain",
        "Estudar agentes",
        "Criar projeto com Ollama"
    ]

    return "Tarefas cadastradas:\n- " + "\n- ".join(tarefas)


## 7. Registrando as ferramentas


In [ ]:
tools = [
    consultar_horario,
    criar_tarefa,
    listar_tarefas
]


## 8. Criando o agente

`create_react_agent` cria um agente que pode utilizar o modelo e as ferramentas fornecidas.

```text
Pergunta
   ↓
Qwen
   ↓
Escolha da Tool
   ↓
Execução
   ↓
Resultado
```


In [ ]:
agent = create_react_agent(
    model=llm,
    tools=tools
)


## 9. Enviando uma pergunta

O agente recebe uma lista de mensagens. Neste exemplo enviamos uma `HumanMessage`.


In [ ]:
def perguntar(pergunta: str):
    resultado = agent.invoke({
        "messages": [
            HumanMessage(content=pergunta)
        ]
    })

    return resultado["messages"][-1].content


## 10. Teste: consultar horário

Pergunta:

```text
Que horas são?
```

O agente deve identificar que precisa utilizar `consultar_horario`.


In [ ]:
print(perguntar("Que horas são?"))


## 11. Teste: criar tarefa

Agora a intenção é diferente:

```text
Crie uma tarefa chamada estudar agentes de IA.
```

O agente precisa escolher `criar_tarefa` e extrair o título.


In [ ]:
print(perguntar("Crie uma tarefa chamada estudar agentes de IA."))


## 12. Teste: listar tarefas


In [ ]:
print(perguntar("Quais tarefas eu tenho?"))


## 13. O ponto principal

Não precisamos escrever uma sequência de `if/elif` para decidir qual função chamar.

Em vez disso:

```text
Pergunta
   ↓
LLM interpreta
   ↓
Escolhe Tool
   ↓
Executa Tool
```

Esse é o papel do agente.


## 14. Agente x lógica tradicional

### Tradicional

```text
if / else
   ↓
regra programada
   ↓
ação
```

### Agente

```text
Pergunta
   ↓
LLM interpreta
   ↓
escolhe ferramenta
   ↓
ação
```

Na prática, aplicações reais podem combinar agentes com regras determinísticas, validações e controles de segurança.


## 15. Mais de uma Tool

Uma solicitação pode exigir várias ações.

Por exemplo:

```text
"Veja minhas tarefas e crie uma nova baseada nelas."
```

Um fluxo possível:

```text
listar_tarefas()
      ↓
analisar resultado
      ↓
criar_tarefa(...)
```

O agente pode orquestrar múltiplas etapas.


## 16. Como isso se encaixa na Nexa AI?

Uma arquitetura possível:

```text
Usuário
   ↓
Central IA
   ↓
AI Service
   ↓
Agente
   ├── criar_tarefa()
   ├── listar_tarefas()
   ├── editar_tarefa()
   ├── excluir_tarefa()
   ├── organizar_agenda()
   └── analisar_produtividade()
```

As Tools podem chamar APIs reais dos seus microsserviços em vez de acessar o banco diretamente.


## 17. Cuidados com agentes

Agentes podem executar ações reais, então uma aplicação precisa de:

- validação dos argumentos;
- autenticação e autorização;
- confirmação para ações sensíveis;
- tratamento de erros;
- logs;
- observabilidade;
- limites de execução;
- idempotência quando necessário.

O agente pode escolher a ferramenta, mas a aplicação continua responsável pela segurança.
